In [210]:
import pandas as pd

def carregar_bases():
    '''
    Params: None
    
    Return: Dataframes list
    '''
    # Lista com caminhos dos arquivos de bases
    df_geoPlan = pd.read_excel(r"bases\GeoPlan.xlsx", usecols=['ID_ESTACAO', 'TIPO_ESTACAO_GEOPLAN'])
    df_infraGest = pd.read_excel(r"bases\InfraGest.xlsx", usecols=['PONTO_ER', 'STATUS_INFRAGEST'])
    df_sisLic = pd.read_excel(r"bases\SisLic.xlsx", usecols=['CODIGO_ER', 'STATUS_SISLIC'])
    df_connectedPrevious = pd.read_excel(r"bases\t_conectados_anterior.xlsx")
    df_connectedCurrent = pd.read_excel(r"bases\t_conectados_atual.xlsx", usecols=['ID_ESTACAO'])

    # Aba 1 do arquivo AuditReport
    df_auditReport = pd.concat(
        pd.read_excel(r"bases\AuditReport.xlsx", sheet_name=None, usecols=['CODIGO_ER 1', 'TIPO_DE_PONTO 62', 'LATITUDE 25', 'LONGITUDE 27']).values(),
        ignore_index=True
    )
    
    # Faz merge do connectedCurrent com conectadosAntigo
    df_connected = df_connectedCurrent.merge(df_connectedPrevious, on='ID_ESTACAO', how='left')

    return df_geoPlan, df_infraGest, df_sisLic, df_connected, df_auditReport

df_geoPlan, df_infraGest, df_sisLic, df_connected, df_auditReport = carregar_bases()

In [211]:
def juntar_bases(df_geoPlan, df_infraGest, df_sisLic, df_connected, df_auditReport):
    '''
    '''
    # juntar bases conectados
    df_connected = (
        df_connected
        .merge(df_geoPlan, on= 'ID_ESTACAO', how= 'left', suffixes=("", '_2'))
        .merge(df_infraGest, left_on= 'ID_ESTACAO', right_on= 'PONTO_ER', how= 'left', suffixes=("", '_2'))
        .merge(df_auditReport, left_on= 'ID_ESTACAO', right_on= 'CODIGO_ER 1', how= 'left', suffixes=("", '_2'))
        .drop(columns=['PONTO_ER', 'CODIGO_ER 1'], errors='ignore')
        )
    
    # Adiciona prefixo AUX_ nas colunas auxiliares
    df_connected = df_connected.rename(columns={
        'TIPO_ESTACAO_GEOPLAN_2': 'AUX_TIPO_ESTACAO_GEOPLAN', 
        'STATUS_INFRAGEST_2': 'AUX_STATUS_INFRAGEST',
        'TIPO_DE_PONTO 62_2': 'AUX_TIPO_DE_PONTO',
        'LATITUDE 25': 'AUX_LATITUDE',
        'LONGITUDE 27': 'AUX_LONGITUDE'
        })
    
    return df_connected, df_sisLic

df_connected, df_sisLic = juntar_bases(df_geoPlan, df_infraGest, df_sisLic, df_connected, df_auditReport)

In [213]:
def tratar_geoPlan(df_connected):
    # Todos os valores nulos em ['AUX_TIPO_ESTACAO_GEOPLAN'] recebem o valor A DEFINIR.
    df_connected['AUX_TIPO_ESTACAO_GEOPLAN'] = df_connected['AUX_TIPO_ESTACAO_GEOPLAN'].fillna('A DEFINIR')
    
    # Se o valor na coluna [TIPO_ESTACAO_GEOPLAN] for igual a A DEFINIR, mas não em [AUX_TIPO_ESTACAO_GEOPLAN], então [TIPO_ESTACAO_GEOPLAN] é igual a [AUX_TIPO_ESTACAO_GEOPLAN]
    FILTRO = (df_connected['TIPO_ESTACAO_GEOPLAN'] == "A DEFINIR") & (df_connected['AUX_TIPO_ESTACAO_GEOPLAN'] != "A DEFINIR")
    df_connected.loc[FILTRO, 'TIPO_ESTACAO_GEOPLAN'] = df_connected['AUX_TIPO_ESTACAO_GEOPLAN']

    return df_connected

df_connected = tratar_geoPlan(df_connected)

display(df_connected)

,ID_ESTACAO,TIPO_ESTACAO_GEOPLAN,LATITUDE,LONGITUDE,STATUS_SISLIC,SISTEMA_ER,MES_REFERENCIA,AUX_TIPO_ESTACAO_GEOPLAN,STATUS_INFRAGEST,TIPO_DE_PONTO 62,AUX_LATITUDE,AUX_LONGITUDE
0,EV-10000,A DEFINIR,0,0,PENDENTE,DESCONHECIDO,02/2026,A DEFINIR,CONECTADO,COBERTURA,-23.507604,-46.667658
1,EV-10001,COMERCIAL,0,0,PENDENTE,DESCONHECIDO,02/2026,COMERCIAL,CONECTADO,INTERNO,-23.524531,-46.622855
2,EV-10002,PONTO ECOLÓGICO,0,0,PENDENTE,DESCONHECIDO,02/2026,PONTO ECOLÓGICO,CONECTADO,PONTO ECOLÓGICO,-23.579398,-46.615521
3,EV-10003,COMERCIAL,0,0,PENDENTE,DESCONHECIDO,02/2026,COMERCIAL,CONECTADO,INTERNO,-23.517086,-46.676691
4,EV-10004,INTERNO,0,0,PENDENTE,DESCONHECIDO,02/2026,INTERNO,CONECTADO,SHOPPING,-23.511403,-46.646886
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,EV-19995,SUPERMERCADO,0,0,PENDENTE,DESCONHECIDO,02/2026,NOVO_TERRENO,CONECTADO,A DEFINIR,-23.507749,-46.671740
9996,EV-19996,COBERTURA,0,0,PENDENTE,DESCONHECIDO,02/2026,COMERCIAL,CONECTADO,COMERCIAL,-23.502459,-46.660035
9997,EV-19997,SUPERMERCADO,0,0,PENDENTE,DESCONHECIDO,02/2026,INTERNO,CONECTADO,INTERNO,-23.514939,-46.627055
9998,EV-19998,NOVO_TERRENO,0,0,PENDENTE,DESCONHECIDO,02/2026,SHOPPING,CONECTADO,A DEFINIR,-23.593835,-46.678561


In [88]:
def tratar_sisLic():
        # tratar duplicidade

        # Caso encontre alguma chave com STATUS_SISLIC = APROVADO, exclui as outras duplicadas.
        # Se não tiver APROVADO, busca EM ANÁLISE e exclui as duplicadas restantes.

        # duplicated_keys = df_sisLic['CODIGO_ER'].duplicated()
        # unique_keys = ~df_sisLic['CODIGO_ER'].duplicated()

        # print(duplicated_keys)
        
        pass

In [89]:
def tratar_infraGest():
        pass

In [90]:
def classificar_ERs():
        pass

In [91]:
def definir_faturamento():
        pass

In [ ]:
def main():
    print("Carregando bases...")
    df_geoPlan, df_infraGest, df_sisLic, df_conectados, df_report = carregar_bases()

    # juntar as colunas de todas as bases e transformar tudo em um. limpar sislic antes
    print("Juntando bases...")
    df_conectados, df_sisLic = juntar_bases(df_geoPlan, df_infraGest, df_sisLic, df_conectados, df_report)

    # 2 - Regras da base GeoPlan:
    #     2.1 - Todos os valores nulos em geoplan['TIPO_ESTACAO_GEOPLAN'] recebem o valor A DEFINIR.
    #     2.2 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] e audit_report['AUX_TIPO_PONTO_AUDIT'] tiverem o mesmo valor, nada é feito.
    #     2.3 - Se o valor na coluna geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a A DEFINIR, mas não em audit_report['AUX_TIPO_PONTO_AUDIT'], então geoplan['TIPO_ESTACAO_GEOPLAN'] assume o valor de audit_report['AUX_TIPO_PONTO_AUDIT'].
    #     2.4 - Mantém os valores de audit_report['AUX_LATITUDE_AUDIT'] e audit_report['AUX_LONGITUDE_AUDIT'] e o que estiver vazio recebe as coordenadas de geoplan['LATITUDE'] e geoplan['LONGITUDE'] genéricas.
    tratar_geoPlan()

    # 3 - Regras da base SisLic:
    #     ** sislic tem chaves repetidas em sislic['CODIGO_ER'] com valores de licença diferentes que não devem ser excluídas de imediato.
    #     Antes da comparação:
    #         Caso encontre alguma chave com sislic['STATUS_SISLIC'] = APROVADO, exclui as outras chaves duplicadas.
    #         Se não tiver APROVADO, busca EM ANÁLISE e exclui as chaves duplicadas restantes.

    #     3.1 - Se o valor na coluna base_consolidada['STATUS_SISLIC'] e sislic['AUX_STATUS_SISLIC'] forem iguais, nada é feito.
    #     3.2 - Se o valor na coluna base_consolidada['STATUS_SISLIC'] for igual a APROVADO e sislic['AUX_STATUS_SISLIC'] não for APROVADO, envia para a tabela de validação.
    #     3.3 - Se o valor na coluna base_consolidada['STATUS_SISLIC'] for diferente de APROVADO, substitui o valor de base_consolidada['STATUS_SISLIC'] pelo valor de sislic['AUX_STATUS_SISLIC'].
    tratar_sisLic()

    # 4 - Regras da base InfraGest:
    #     4.1 - Se o valor em base_consolidada['SISTEMA_ER'] for igual a CONECTADO e infragest['AUX_STATUS_INFRAGEST'] for igual a CONECTADO, nada é feito.
    #     4.2 - Se o valor em base_consolidada['SISTEMA_ER'] estiver vazio, recebe o valor de infragest['AUX_STATUS_INFRAGEST'].
    #     4.3 - Se o valor em base_consolidada['SISTEMA_ER'] for igual a CONECTADO e infragest['AUX_STATUS_INFRAGEST'] for diferente de CONECTADO, envia para validação.
    #     4.4 - Se o valor em base_consolidada['SISTEMA_ER'] estiver vazio e o valor de infragest['AUX_STATUS_INFRAGEST'] também, então base_consolidada['SISTEMA_ER'] é igual a INFRA NÃO CAPACITADA.
    tratar_infraGest()

    # 5 - Regras para classificação de ERs:
    #     5.1 - Se o valor na coluna base_consolidada['STATUS E-MAIL'] estiver preenchido, mantém a informação de base_consolidada['STATUS CONSOLIDADO'].
    #     5.2 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a SHOPPING ou SUPERMERCADO ou PONTO ECOLÓGICO, então base_consolidada['STATUS CONSOLIDADO'] é igual a ESTAÇÕES SUSTENTÁVEIS (ENERGIA SOLAR).
    #     5.3 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a INTERNO, então base_consolidada['STATUS CONSOLIDADO'] é igual a CARREGADOR LENTO - INTERNO.
    #     5.4 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a COBERTURA, então base_consolidada['STATUS CONSOLIDADO'] é igual a ESTAÇÃO EM COBERTURA.
    #     5.5 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a A DEFINIR ou NOVO_TERRENO ou COMERCIAL e sislic['STATUS_SISLIC'] for igual a APROVADO, então base_consolidada['STATUS CONSOLIDADO'] é igual a OPERACIONAL.
    #     5.6 - Se geoplan['TIPO_ESTACAO_GEOPLAN'] for igual a A DEFINIR ou NOVO_TERRENO ou COMERCIAL e sislic['STATUS_SISLIC'] for igual a NÃO TEM LICENÇA ou PENDENTE INSTALAÇÃO, então base_consolidada['STATUS CONSOLIDADO'] mantém os status atuais e nos campos vazios muda para "SEM CONEXÃO DE REDE".
    classificar_ERs()

    # 6 - Regras da coluna STATUS_FATURAMENTO:
    #     6.1 - Se base_consolidada['STATUS CONSOLIDADO'] for igual a OPERACIONAL e infragest['SISTEMA_ER'] for igual a CONECTADO ou PENDENTE ATIVAÇÃO, então base_consolidada['STATUS_FATURAMENTO'] é igual a 1.
    #     6.2 - Se base_consolidada['STATUS CONSOLIDADO'] for igual a OPERACIONAL e infragest['SISTEMA_ER'] for igual a INFRA NÃO CAPACITADA, então base_consolidada['STATUS_FATURAMENTO'] é igual a 0.
    #     6.3 - Se qualquer valor não bater com uma das duas condições anteriores, então base_consolidada['STATUS_FATURAMENTO'] é igual a 2.
    definir_faturamento()

    pass